# Chapter 4. 유용한 선형회귀 (~4.5 양수성과 외삽)

## 1. 선형회귀의 필요성

- 회귀분석은 인과추론의 핵심이자 가장 많이 사용하는 방법
- 또한 회귀분석은 대부분의 패널데이터 방법(이중차분법(DID), 이원고정효과(two-way fixed effects), 머신러닝 방법(이중/편항 제거 머신러닝), 그리고 다른 식별 기법(도구 변수, 불연속 설계) 등 응용 방법론의 주요 구성 요소

$ATE = sum_x {E[Y | T = 1, X = x]P(X = x) - E[Y | T = 0, X = x]P(X = x)}$
- 처치가 무작위 배정된 것처럼 변수를 보정하는 보정공식
- 여기에 조건부 독립성 가정이 만족되면 인과효과를 식별할 수 있음
- 그러나, 많은 변수 그리고 여러 그룹으로 구성된 데이터를 다룰 경우 각 셀에서 ATE를 추정하고 각 결과의 평균값을 구하려면 엄청난 양의 데이터 필요
    - 차원의 저주(curse of dimensionality)
- 차원의 저주 때문에 공변량이 많을 때 보정 공식을 그대로 적용하면 데이터 희소성 문제를 겪을 수 있음  

차원의 저주에서 어떻게 벗어날 수 있을까?
- 잠재적 결과를 선형회귀 같은 방식으로 모델링할 수 있다고 가정하고, X로 정의된 각 셀을 **내삽(interpolate)** 하고 **외삽(extrapolate)**.

=================
#### 내삽 (Interpolation)
**관측된 데이터 포인트 사이의 값을 추정하는 것**입니다. 
- 예: X = [1, 2, 5]에서 데이터가 관측되었을 때, X = 3이나 X = 4에서의 값을 추정하는 것
- 기존 데이터의 범위 **내부**에서 값을 예측
- 비교적 안정적이고 신뢰할 수 있는 추정

#### 외삽 (Extrapolation)
**관측된 데이터 범위를 벗어난 값을 추정하는 것**입니다.
- 예: X = [1, 2, 5]에서 데이터가 관측되었을 때, X = 7이나 X = 10에서의 값을 추정하는 것
- 기존 데이터의 범위 **외부**에서 값을 예측
- 내삽보다 불확실성이 높고 위험할 수 있음

#### 인과추론 맥락에서의 의미
1. **문제**: 공변량 X가 많을 때, 각 X의 조합(셀)마다 충분한 데이터가 없을 수 있음
2. **해결**: 선형회귀 등의 모델을 사용하여:
   - **내삽**: 데이터가 희소한 X 조합에 대해, 주변 관측값을 이용해 잠재적 결과를 추정
   - **외삽**: 전혀 관측되지 않은 X 조합에 대해서도 모델을 통해 잠재적 결과를 추정

3. **장점**: 모든 X 조합에 대해 직접 데이터를 수집할 필요 없이, 모델을 통해 효율적으로 ATE를 추정할 수 있음

예를 들어, 나이와 소득 수준에 따른 처치효과를 추정할 때, "나이 35세, 소득 5000만원"인 그룹의 데이터가 없어도 주변 데이터(나이 30세/40세, 소득 4000만원/6000만원)를 활용해 이 그룹의 잠재적 결과를 추정하는 것.

## 2. 회귀분석을 통한 보정
신용 한도가 채무불이행률에 미치는 영향을 추정하는 사례(신용한도와 채무불이행률 사이에 음의 상관관계 존재한다고 가정)
- 신용 가치를 나타낼 수 있는 고객 특성(월급, 신용점수, 현재 회사 근무기간, 결혼여부 등), 해당 고객에게 주어진 신용한도(처치), 고객의 채무불이행 여부
- 신용한도에 따른 평균 채무불이행률과 함께 적합된 회귀선을 시각화했더니 -> 음의 추세 명확
    - (교란 요인)
        - 신용점수 높은 사람 → 은행이 한도를 많이 줌 + 원래 성실해서 불이행 안 함
        - 신용점수 낮은 사람 → 은행이 한도를 적게 줌 + 원래 불이행 위험 높음
        - 월급 많은 사람 → 한도도 높고 + 돈 갚을 능력이 있음
        - 월급 적은 사람 → 한도도 낮고 + 돈 갚기 어려움
        - 즉, 신용한도 자체가 채무불이행을 줄이는 게 아니라, 원래 신용이 좋은 사람들이 한도도 많이 받는 것!
- 이 편향을 보정하기 위해.
    - **이론적으로는**
        - (1) 모든 교란 요인에 따라 데이터를 나누고
        - (2) 나눈 각 그룹 내에서 채무불이행률을 신용 한도에 회귀하고
        - (3) 기울기 매개변수 추출을 통해 결과의 평균을 구함
        - But, 그룹이 너무 많아서 각 그룹에 사람이 몇 명 안 되거나 아예 없을 가능성 높음 → 차원의 저주
    - 교란요인을 직접 보정하는 대신, OLS로 추정할 모델에 단순히 교란 요인을 추가(신용점수와 임금을 교란요인으로 모델에 추가)
        - 채무불이행 = 상수 
           + + (신용한도의 효과) × 신용한도
           + + (월급의 효과) × 월급  
           + + (신용점수의 효과) × 신용점수
           + + 오차
        - $Default_i = \Beta_0 + \Beta_1line_i + \theta_1wage_i + \theta_2creditScore1_i + \theta_3creditScore2_i + e_i$
        - 이렇게 하면, 월급과 신용점수가 같은 사람들끼리 비교하는 효과.
        - 즉, 다른 조건이 동일할 때 신용한도만의 순수한 효과를 측정

**1. 교란 요인을 넣지 않은 모델(편향된 모델)**
- 채무불이행 = β₀ + β₁ × 신용한도 + 오차
- 예시 상황:
    - A씨: 신용한도 1000만원 → 채무불이행 확률 5%
    - B씨: 신용한도 500만원 → 채무불이행 확률 15%
- 잘못된 해석:
    - "신용한도를 500만원 늘리면 채무불이행이 10% 감소한다!" 
- 진짜 문제:
    - A씨는 원래 월급 500만원, 신용점수 800점 (우량 고객)
    - B씨는 원래 월급 200만원, 신용점수 600점 (위험 고객)
    → 신용한도의 효과인지, 원래 신용도 차이인지 구분 못함!

**2. 교란 요인을 넣은 모델(보정된 모델)**
- 교란요인을 넣으면 각 변수의 독립적인 효과를 측정할 수 있음
- 채무불이행 = β₀ + β₁×신용한도 + θ₁×월급 + θ₂×신용점수1 + θ₃×신용점수2 + 오차
    - β₀ (베타 제로) = 모든 변수가 0일 때의 채무불이행률 (현실에서는 큰 의미 없음)
    - β₁ (베타 원) = 우리가 정말 알고 싶은 것! 
        - 다른 조건이 모두 같을 때, 신용한도 100만원당 채무불이행률 변화
        - 예: β₁ = -0.02 → 한도 100만원 증가 시 불이행률 2% 감소
    - θ₁ (세타 원) = 월급의 효과
        - 신용한도와 신용점수가 같을 때, 월급 100만원당 채무불이행률 변화
    - θ₂, θ₃ (세타 투, 쓰리) = 신용점수의 효과
        - 신용점수 구간별 채무불이행률 차이

## 3. 프리슈-워-로벨 정리와 직교화

- Frisch-Waugh-Lovell 스타일의 직교화(잔차화)는 가장 먼저 사용할 수 있는 편향 제거 기법
- FWL 정리에 따른 추정과정
    - (1) 편향 제거 단계 : 처치 T를 교란 요인 X에 회귀하여 처치 잔차 $\tilde{T}$ = T - $\hat{T}$ 를 구함
    - (2) 잡음 제거 단계 : 결과 Y를 교란 요인 X에 대해 회귀하여 결과 잔차 $\tilde{Y}$ = Y - $\hat{Y}$를 구함
    - (3) 결과 모델 단계 : 결과 잔차 $\tilde{Y}$를 처치 잔차 $\tilde{T}$에 대해 회귀하여 T가 Y에 미치는 인과효과 추정값을 구함

=============================================================

#### 1. 편향 제거 단계
- 처음에는 교란편향의 영향으로 신용 한도에 따라 채무불이행률이 감소하는 추세를 보였음
- FWL 정리에 따르면 교란 요인으로부터 처치인 신용 한도를 예측하는 회귀모델을 적합시켜 데이터의 편향을 제거할 수 있음
- 그 다음, 이 모델로부터 신용 한도에 대한 잔차를 구함. 이 잔차는 편향 제거 모델에 사용된 변수와는 상관관계가 없는 버전의 처치로 볼 수 있음
    - 예측값을 생성한 변수와 직교하기 때문
- 이를 통해 모든 교란 요인이 편향 제거 모델에 포함된다면 신용 한도가 채무불이행률에 미치는 인과적 영향에 대해 편향되지 않은 추정값을 얻을 수 있음
- 편향 제거된 버전의 신용 한도와 채무불이행률을 시각화하면 두 변수 사이의 관계가 하향선을 그리지 않음

#### 2. 잡음 제거 단계
- 인과효과 추정을 정확하게 하려면 편향 제거 단계가 중요하며 잡음 제거 단계는 그만큼 중요하진 않지만 포함하면 좋음
- 잡음을 제거한다고 처치효과의 추정값이 바뀌지는 않지만 분산을 줄일 수 있음. 이 단계에선 결과를 처치가 아닌 공변량에 대해 회귀함

#### 3. 결과 모델 단계
- FWL 정리의 마지막 단계인 결과 모델에선 두 잔차 $\tilde{Y}$와 $\tilde{T}$를 이용해 단순히 $\tilde{Y}$를 $\tilde{T}$에 대해 회귀하면 됨
- 편향 제거 단계에서 얻은 매개변수 추정값은 신용한도와 다른 모든 공변량을 사용하여 회귀했을 때와 완전히 동일
- 또한, 표준오차와 p값도 이제 모든 변수를 포함해 처음 모델을 실행했을 때와 같음

===============================================================  
원본 데이터 (교란 요인으로 오염됨)  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  
신용한도 ■■■■■■■■■■ (높음)  →  채무불이행 ■■ (낮음)  
         ↑ 상관관계 있음 ↑  
    하지만 인과관계는...?  


[1단계] 💨 편향 제거 (처치 잔차화)  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  
신용한도 = f(소득, 나이, 신용점수, ...)  
         ⬇️ 예측  
    예측된 신용한도 ■■■■■■■■  
         ⬇️ 빼기  
    잔차 (T̃) = 실제 - 예측  
    → "교란 요인으로 설명 안 되는 순수한 신용한도 변동"  
  

[2단계] 🧹 잡음 제거 (결과 잔차화)  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  
채무불이행 = f(소득, 나이, 신용점수, ...)  
         ⬇️ 예측  
    예측된 채무불이행 ■■■  
         ⬇️ 빼기  
    잔차 (Ỹ) = 실제 - 예측  
    → "교란 요인으로 설명 안 되는 순수한 채무불이행 변동"  
  

[3단계] 🎯 결과 모델  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  
Ỹ (결과 잔차) = β × T̃ (처치 잔차)  

이제 β가 진짜 인과효과! 

=============================================================

편향 제거: 교란 요인의 영향을 빼서 "순수한" 처치 변동만 추출  
잡음 제거: 결과에서도 교란 요인의 영향을 빼서 추정의 정확도 ↑  
결과 모델: 두 잔차의 관계 = 순수한 인과효과  

=============================================================

잔차들은 교란 요인과 직교(상관관계 = 0)  
3단계로 나눠서 하든, 한 번에 다중회귀 하든 결과는 동일!  
잡음 제거는 선택사항이지만, 하면 표준오차가 줄어듦 (더 정확)  

## 4. 양수성과 외삽

- 회귀분석은 잠재적 결과를 모수적(parametric)으로 모델링하므로 데이터의 처치 범위 이외에 대해서도 외삽 가능
    - 모수적 모델링이란?
        - Y = β₀ + β₁X + β₂Z + ... + 오차
        - 이런 수식 하에 실제로 관측한 X값 범위를 넘어서도 예측 가능
    - 처치 범위 이외에 대해서도 외삽 가능의 의미
        - 신용카드 예시:
        - 실제 데이터: 신용한도 100만원 ~ 500만원 사이의 고객만 관측됨
        - 회귀모델: 채무불이행 = β₀ + β₁×신용한도 + ...
        - 외삽 능력: 이 모델로 신용한도 700만원, 1000만원일 때도 예측 가능!
    - 왜 가능한가?
        - 선형회귀는 "직선"이라는 전역적 패턴을 가정
        - 관측된 범위에서 학습한 관계가 범위 밖에도 적용된다고 가정
- 그러나 지나친 외삽은 항상 위험함.
    - 위험한 이유 3가지
        - (1) 관계가 바뀔 수 있음
            - 관측 범위 (100~500만원): 신용한도 ↑ → 채무불이행 ↓ (선형 관계)
            - 외삽 범위 (1000~2000만원): 신용한도가 너무 높으면? → 과소비 유발 → 오히려 채무불이행 ↑ (비선형!)

        - (2) 데이터가 없는 영역은 불확실
            - 실제 관측 : 신용한도(100, 200, 300, 400, 500), 불이행률(15%, 12%, 10%, 8%, 6%)
            - 외삽 예측 : 신용한도 1000만원, 불이행률 : 0% ? (음수는 불가능) ← 말이 안 됨

        - (3) 교란 요인의 분포가 달라짐
            - 관측된 범위:
                - 신용한도 100만원 → 보통 월급 200만원
                - 신용한도 500만원 → 보통 월급 500만원

            - 외삽 범위:  
                - 신용한도 2000만원 → 월급이 얼마인 사람?  
                → 이런 사람들의 데이터가 없으면  
                → 교란 요인 보정도 의미 없어짐

- 안전한 외삽
    - 관측: 신용한도 100~500만원
    - 외삽: 신용한도 600만원 예측 ✅  
    → 패턴이 유지될 가능성 높음
- 위험한 외삽
    - 관측: 신용한도 100~500만원  
    - 외삽: 신용한도 5000만원 예측 ❌  
        → 완전히 다른 고객층, 다른 행동 패턴  
        → 예측 신뢰도 매우 낮음  

===========================================================================  

인과추론의 핵심 가정: 양수성(Positivity)
- 모든 X 값에서 T=0과 T=1을 모두 관측해야 함
- 외삽으로 이 문제를 "우회"할 수 있지만 가정이 틀리면 인과효과 추정도 틀림